# PV Project

### Import Libraries

In [15]:
import duckdb
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path
import warnings
from scipy.special import gammaln, digamma, expit, logit
from scipy.optimize import minimize
from scipy.stats import gamma as G

#### Establish the DB connection

In [16]:
# Establishing connection wit the database

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DB_PATH = PROJECT_ROOT / "data" / "faers_db.duckdb"

con = duckdb.connect(str(DB_PATH), read_only=False) # read_only=True for exploration, and read_only=False only when creating tables.

#### Identify the tables

In [20]:
pd.set_option("display.max_columns", 60, "display.width", 220, "display.max_colwidth", 50)
print(con.sql("SHOW TABLES").df())

                     name
0               case_base
1               case_drug
2           case_exposure
3              case_month
4            case_outcome
5                 case_pt
6             case_source
7              clean_demo
8                 demo_us
9             drug_detail
10               exposure
11             faers_flat
12                    m_N
13                   m_ev
14                  m_grp
15                 m_pair
16                 months
17            pair_counts
18              pair_grid
19             pair_month
20                ps_case
21          ps_case_event
22                ps_drug
23               raw_demo
24               raw_indi
25               raw_reac
26               raw_ther
27               ref_drug
28       ref_known_signal
29          signal_labels
30              signal_ts
31       study_case_event
32            study_cases
33        study_drug_rows
34   v_ingredient_summary
35   v_known_signal_check
36  v_known_signal_family
37        v_

In [21]:
pair_counts = con.sql("SELECT * FROM pair_counts").df()
pair_month= con.sql ("SELECT * FROM pair_month").df()
pair_grid= con.sql ("SELECT * FROM pair_grid").df()
v_signals= con.sql ("SELECT * FROM v_signals").df()
ps_case_event= con.sql ("SELECT * FROM ps_case_event").df()
m_pair= con.sql ("SELECT * FROM m_pair").df()

In [ ]:
m_pair

In [ ]:
ps_case_event.describe()
ps_case_event.columns.tolist()
ps_case_event.head(3)

### Import the files

In [25]:
print("Current Working Directory:", os.getcwd())
print(os.listdir())

Current Working Directory: /Users/shymashameer/Desktop/Final Project/faers/notebooks
['.DS_Store', '05_Disprotionality Analysis.ipynb', '05_Timeseries.ipynb', '01_Loading Data.ipynb', '04_Forecasting.ipynb', '02_Initial EDA.ipynb', '.ipynb_checkpoints', '03_Exploration.ipynb', 'outputs']


In [26]:
sig= pd.read_csv("outputs/signal_identification_v1/sig_final.csv")
counts_by_drug = pd.read_csv("outputs/signal_identification_v1/counts_by_drug.csv")
res_raw = pd.read_csv("outputs/signal_identification_v1/res_raw.csv")
run_settings = pd.read_csv("outputs/signal_identification_v1/run_settings.csv")
strong_signals = pd.read_csv("outputs/signal_identification_v1/strong_signals.csv")
watchlist = pd.read_csv("outputs/signal_identification_v1/watchlist.csv")

In [27]:
counts_by_drug

,ingredient,signal,strong
0,canagliflozin,243,191
1,dapagliflozin,161,114
2,dulaglutide,84,59
3,empagliflozin,154,113
4,ertugliflozin,15,11
5,liraglutide,112,80
6,semaglutide,66,45


In [28]:
# Check if correctly loaded
print(sig.shape)                                    # expect 9251 rows
print(sig.dtypes[["is_noise","signal","strong"]])   # should be bool, not object or int
print(sig.tier.value_counts())                      # noise / strong / signal-not-strong / not flagged
nn = sig[~sig.is_noise]
print(len(nn), nn.signal.sum(), nn.strong.sum())    # analysed, signals, strong
print(nn.groupby("ingredient")[["signal","strong"]].sum())
#print(pd.read_csv(out / "run_settings.csv").to_string())

(9251, 32)
is_noise    bool
signal      bool
strong      bool
dtype: object
tier
3 Not flagged           7905
1 Strong                 613
0 Noise (excluded)       511
2 Signal, not strong     222
Name: count, dtype: int64
8740 835 613
               signal  strong
ingredient                   
canagliflozin     243     191
dapagliflozin     161     114
dulaglutide        84      59
empagliflozin     154     113
ertugliflozin      15      11
liraglutide       112      80
semaglutide        66      45


### Further Analysis

In [22]:
print(con.sql("SELECT * FROM signal_labels LIMIT 5").df().to_string())
print(con.sql("SELECT * FROM ref_known_signal").df().to_string())
print(con.sql("SELECT * FROM pair_counts LIMIT 5").df().to_string())

      ingredient                   pt      onset    status
0  canagliflozin    AMYLASE INCREASED        NaT     never
1  canagliflozin           ANGIOPATHY        NaT     never
2  canagliflozin           ARTHRALGIA        NaT     never
3  canagliflozin  ARTHRITIS BACTERIAL 2018-02-01  emerging
4  canagliflozin  BACTERIAL INFECTION        NaT     never
      ingredient      pt_regex   reg_date                                   source
0  dapagliflozin  KETOACIDOSIS 2015-05-15         FDA DSC: SGLT2i and ketoacidosis
1  empagliflozin  KETOACIDOSIS 2015-05-15         FDA DSC: SGLT2i and ketoacidosis
2  canagliflozin  KETOACIDOSIS 2015-05-15         FDA DSC: SGLT2i and ketoacidosis
3  canagliflozin    AMPUTATION 2016-05-18  FDA DSC: interim CANVAS amputation risk
4  dapagliflozin      FOURNIER 2018-08-29  FDA DSC: SGLT2i and Fournier's gangrene
5  empagliflozin      FOURNIER 2018-08-29  FDA DSC: SGLT2i and Fournier's gangrene
6  canagliflozin      FOURNIER 2018-08-29  FDA DSC: SGLT2i and Fo

In [23]:
# A. coverage
print(con.sql("""SELECT COUNT(*) pairs FROM (SELECT DISTINCT ingredient, pt FROM pair_month)""").df())
print(con.sql("""SELECT COUNT(*) pairs, MIN(a) min_a, MIN(m), MAX(m) FROM
                 (SELECT ingredient, pt, a, m FROM signal_ts)""").df())

# B. does the last month of pair_month reproduce pair_counts?
print(con.sql("""
WITH l AS (SELECT * FROM pair_month WHERE m = (SELECT MAX(m) FROM pair_month))
SELECT COUNT(*) n_joined,
       SUM((l.a <> f.a)::INT)                     a_diff,
       SUM((l.n_exp <> f.n_exp)::INT)             exp_diff,
       SUM((l.n_excl <> f.n_excl)::INT)           excl_diff,
       SUM((l.n_event_all <> f.n_event_all)::INT) ev_all_diff,
       SUM((l.n_event_excl <> f.n_event_excl)::INT) ev_excl_diff,
       SUM((l.N <> f.N)::INT)                     N_diff
FROM l JOIN pair_counts f USING (ingredient, pt)""").df())

# C. does the last month of signal_ts reproduce the final signal count?
print(con.sql("""
SELECT (SELECT COUNT(*) FROM signal_ts WHERE signal AND m = (SELECT MAX(m) FROM signal_ts)) AS ts_last_month,
       (SELECT COUNT(*) FROM v_pair_stats WHERE signal) AS final_signal""").df())

# D. how labels relate to the final state
print(con.sql("""
SELECT l.status, COUNT(*) n, SUM(COALESCE(s.signal, FALSE)::INT) still_signal_at_end
FROM signal_labels l LEFT JOIN v_pair_stats s USING (ingredient, pt)
GROUP BY ALL""").df())

   pairs
0   3870
    pairs  min_a     min(m)     max(m)
0  191758    1.0 2014-01-01 2019-12-01
   n_joined  a_diff  exp_diff  excl_diff  ev_all_diff  ev_excl_diff  N_diff
0      3870     0.0       0.0        0.0          0.0           0.0     0.0
   ts_last_month  final_signal
0            955           955
         status     n  still_signal_at_end
0         never  2732                  0.0
1      emerging   964                821.0
2  pre-existing   174                134.0


### What checks A to D show

All four pass, and they settle several open questions.

* A and B: pair_month is a valid panel base
    * 3,870 pairs is by design, not a gap. From our tier × a-band crosstab, 5,381 of the 9,251 pairs have a < 3, and 9,251 − 5,381 = 3,870. So pair_month and signal_ts contain exactly the pairs with final a ≥ 3. Under min_a = 3, the other pairs can never be flagged, so nothing is lost for the forecasting target. 
    * B shows zero differences in a, n_exp, n_excl, n_event_all, n_event_excl and N across all 3,870 pairs. The last month of pair_month reproduces pair_counts exactly.
* C: signal_ts reproduces our final signals

The last month has 955 signals, equal to the 955 in v_pair_stats. This is the pre-noise Evans ∧ ROR ∧ IC count, matching our per-drug sum (251 + 172 + 132 + 162 + 21 + 136 + 81). The monthly series is therefore consistent with the final analysis for the signal tier.

* D: Ever signalled = 964 + 174 = 1,138. Still signalled at the end = 821 + 134 = 955.
    * The difference is 183 pairs (16%) that crossed the threshold at some month and later fell out.
    * signal_labels covers the same 3,870 pairs, so the universe matches, but it uses the Evans ∧ ROR ∧ IC rule with no EB05 and no final noise list. That is why it can’t serve as the strong-signal target.
    * The 183 flickering pairs argue against a simple “first month signal = true” onset. A pair that crosses once and drops out would count as emerged. Define onset as the first month of a sustained run (for example 3 consecutive months), or require the pair to be signalled at the end. Need to decide this before looking at forecasting results.
    * Pre-existing means onset in 2014 (burn-in). would exclude these from the emergence target and keep them as controls.


In [30]:
e = con.sql("""SELECT ingredient, pt, a,
               n_exp * n_event_all::DOUBLE / N AS E_try FROM pair_counts""").df()
m = e.merge(res_raw[["ingredient","pt","E"]], on=["ingredient","pt"])
print(len(m), (abs(m.E - m.E_try) / m.E).describe())

9251 count    9.251000e+03
mean     2.021144e-03
std      1.168973e-02
min      3.003801e-08
25%      2.323989e-04
50%      4.633229e-04
75%      9.455174e-04
max      4.987384e-01
dtype: float64


In [31]:
cand = con.sql("""
SELECT ingredient, pt,
  n_exp * n_event_all::DOUBLE / N                               AS E1,
  n_exp * (n_event_all - n_event_excl)::DOUBLE / N              AS E2,
  n_exp * n_event_all::DOUBLE / (N - n_excl)                    AS E3,
  n_exp * (n_event_all - n_event_excl)::DOUBLE / (N - n_excl)   AS E4,
  n_exp * (n_event_all - a)::DOUBLE / (N - n_exp)               AS E5
FROM pair_counts""").df()
m = cand.merge(res_raw[["ingredient","pt","E"]], on=["ingredient","pt"])
for k in ["E1","E2","E3","E4","E5"]:
    r = (m.E - m[k]).abs() / m.E
    print(k, "median", r.median(), "max", r.max(), "n>1e-6:", (r > 1e-6).sum())

E1 median 0.0004633228772463229 max 0.49873842204509283 n>1e-6: 9238
E2 median 0.0006856806436353904 max 0.001415088546770406 n>1e-6: 9251
E3 median 0.0003663003663005389 max 0.5000000000000159 n>1e-6: 5440
E4 median 0.0 max 1.1988407130584498e-13 n>1e-6: 0
E5 median 0.00221523311589094 max 1.0 n>1e-6: 9247


In [32]:
print(con.sql("SELECT ingredient, g, SUM(n) cases FROM m_grp GROUP BY ALL ORDER BY 1,2").df())

       ingredient     g    cases
0   canagliflozin  excl   4520.0
1   canagliflozin   exp  18865.0
2   dapagliflozin  excl   2414.0
3   dapagliflozin   exp   4851.0
4     dulaglutide  excl   2490.0
5     dulaglutide   exp  23381.0
6   empagliflozin  excl   3685.0
7   empagliflozin   exp   6407.0
8   ertugliflozin  excl     87.0
9   ertugliflozin   exp    358.0
10    liraglutide  excl   7605.0
11    liraglutide   exp  11604.0
12    semaglutide  excl    348.0
13    semaglutide   exp   2187.0


#### excl is confirmed

The counts match m_grp and pair_counts exactly:

Ingredient	exp_n	excl_n
canagliflozin	18,865	4,520
liraglutide	11,604	7,605

So the definitions are settled:

exp = ingredient is the primary suspect (ps_X)
excl = ingredient is present in another role, i.e. any_X AND NOT ps_X
reference = all other FAERS cases in the 2014-2019 universe (N ≈ 5.37 million)

This is a FAERS-wide comparison with a drug-specific exclusion, not a comparison against other antidiabetics. That differs from how you described it at the start, so decide which one the thesis intends. If it is the FAERS-wide design, the methods text should say so, for example: “Primary-suspect reports for each ingredient were compared against all other FAERS reports (2014-2019), excluding reports in which that ingredient was a non-primary-suspect drug.” An antidiabetic-only comparator would be a separate sensitivity analysis with its own pair_counts.

### Monthly E

In [ ]:
# 1. monthly E (formula E4, which matched res_raw to ~1e-13)
con.sql("""
CREATE OR REPLACE TABLE fc_pair_month_e AS
SELECT ingredient, pt, m, a, n_exp, n_excl, n_event_all, n_event_excl, N,
       n_exp * (n_event_all - n_event_excl)::DOUBLE / (N - n_excl) AS E
FROM pair_month
""")

chk = con.sql("""SELECT ingredient, pt, E FROM fc_pair_month_e
                 WHERE m = (SELECT MAX(m) FROM fc_pair_month_e)""").df()
mm = chk.merge(res_raw[["ingredient","pt","E"]], on=["ingredient","pt"], suffixes=("_ts","_res"))
print(len(mm), ((mm.E_ts - mm.E_res).abs() / mm.E_res).max())   

3870 5.6952114976796085e-15


3,870 rows and a maximum relative difference near 1e-12 means the monthly E reproduces your final E at the last month.

In [36]:
# 2. EBGM / EB05 test on 300 sampled pairs, using the prior from your fit
import numpy as np
from scipy.special import gammaln, digamma
from scipy.stats import gamma
from scipy.optimize import brentq

a1,b1,a2,b2,w = 0.1184,0.0284,0.7926,2.11,0.1355

def eb(n, E):
    def lm(al, be):
        return (gammaln(al+n) - gammaln(al) - gammaln(n+1)
                + al*np.log(be/(be+E)) + n*np.log(E/(be+E)))
    l1, l2 = np.log(w)+lm(a1,b1), np.log(1-w)+lm(a2,b2)
    mx = max(l1, l2)
    q1 = np.exp(l1-mx)/(np.exp(l1-mx)+np.exp(l2-mx)); q2 = 1-q1
    ebgm = np.exp(q1*(digamma(a1+n)-np.log(b1+E)) + q2*(digamma(a2+n)-np.log(b2+E)))
    cdf = lambda x: q1*gamma.cdf(x, a1+n, scale=1/(b1+E)) + q2*gamma.cdf(x, a2+n, scale=1/(b2+E))
    eb05 = brentq(lambda x: cdf(x)-0.05, 1e-12, 1e4)
    return ebgm, eb05

t = res_raw[res_raw.a >= 3].sample(300, random_state=1).copy()
t[["ebgm_t","eb05_t"]] = [eb(r.a, r.E) for r in t.itertuples()]
print((abs(t.ebgm_t-t.EBGM)/t.EBGM).max(), (abs(t.eb05_t-t.EB05)/t.EB05).max())

0.00013372536227838485 0.000260252492234619


both maxima at about 1e-6 or smaller means the standard computation and the weight assignment are right. If not, swap w and 1 - w (set w = 1 - 0.1355) and re-run. If it still fails, paste the function you used for E, EBGM and EB05.

#### Close the DB Connection

In [14]:
con.close()

# Explicitly close any ambient DuckDB connections
try:
    duckdb.close()
    print("✓ Closed ambient DuckDB connection.")
except Exception as e:
    print(f"Notice: {e}")

✓ Closed ambient DuckDB connection.
